In [ ]:
import pandas as pd
import numpy as np
import ast

import os
import matplotlib.pyplot as plt
import seaborn as sns
from plinder_analysis_utils import DockingAnalysisBase, PoseBustersAnalysis, PropertyAnalysis

import statsmodels.formula.api as smf

# Load Data 

In [ ]:
BUST_TEST_COLUMNS = [
    # accuracy #
    "rmsd_≤_2å",
    # chemical validity and consistency #
    "mol_pred_loaded",
    "mol_true_loaded",
    "mol_cond_loaded",
    "sanitization",
    "molecular_formula",
    "molecular_bonds",
    "tetrahedral_chirality",
    "double_bond_stereochemistry",
    # intramolecular validity #
    "bond_lengths",
    "bond_angles",
    "internal_steric_clash",
    "aromatic_ring_flatness",
    "double_bond_flatness",
    "internal_energy",
    # intermolecular validity #
    "minimum_distance_to_protein",
    "minimum_distance_to_organic_cofactors",
    "minimum_distance_to_inorganic_cofactors",
    "volume_overlap_with_protein",
    "volume_overlap_with_organic_cofactors",
    "volume_overlap_with_inorganic_cofactors",
]

In [ ]:
df_combined = pd.read_csv("plinder_set_0_annotated.csv")
# build a boolean mask: drop any row where covalent, ionic or has_ion is True
mask = ~(
    df_combined['ligand_is_covalent'] |
    df_combined['ligand_is_ion'] |
    df_combined['has_ion'] |
    df_combined['ligand_is_cofactor']
)

# filter and reset index
df_combined = df_combined.loc[mask].reset_index(drop=True)
print("Filtered shape:", df_combined.shape)

# Analysis

In [ ]:
# Initialize the analysis object
analysis = PoseBustersAnalysis(df_combined)

In [ ]:
# Find the best pose (lowest RMSD) for each method and protein
best_poses = df_combined.loc[df_combined.groupby(['protein', 'method'])['rmsd'].idxmin()]

# List of methods to analyze
methods = best_poses['method'].unique()

# Calculate pass rate for each method and each PoseBuster test
pass_rates_df = pd.DataFrame(index=BUST_TEST_COLUMNS, columns=methods)
pass_rates_df = pass_rates_df.astype(float)  # Explicitly convert to float type

for method in methods:
    method_poses = best_poses[best_poses['method'] == method]
    for test in BUST_TEST_COLUMNS:
        if test in method_poses.columns:
            # First convert the test column to numeric values if they're boolean
            if pd.api.types.is_bool_dtype(method_poses[test]):
                numeric_values = method_poses[test].astype(float)
            else:
                # Try to convert to numeric, coerce errors to NaN
                numeric_values = pd.to_numeric(method_poses[test], errors='coerce')
                
            # Calculate mean of valid entries
            valid_values = numeric_values.dropna()
            if len(valid_values) > 0:
                pass_rates_df.loc[test, method] = valid_values.mean() * 100
            else:
                pass_rates_df.loc[test, method] = np.nan

# Create a heatmap with better formatting
plt.figure(figsize=(14, 12))
ax = sns.heatmap(pass_rates_df, annot=True, cmap='viridis', fmt='.1f', 
            cbar_kws={'label': 'Pass Rate (%)'})
plt.title('PoseBusters Pass Rate (%) for Best Poses by Method', fontsize=16)
plt.ylabel('PoseBuster Test', fontsize=14)
plt.xlabel('Method', fontsize=14)
plt.tight_layout()
plt.show()

# Group PoseBuster tests into categories for more organized analysis
categories = {
    'Accuracy': ['rmsd_≤_2å'],
    'Chemical Validity': ['mol_pred_loaded', 'mol_true_loaded', 'mol_cond_loaded', 
                        'sanitization', 'molecular_formula', 'molecular_bonds'],
    'Stereochemistry': ['tetrahedral_chirality', 'double_bond_stereochemistry'],
    'Intramolecular Validity': ['bond_lengths', 'bond_angles', 'internal_steric_clash', 
                              'aromatic_ring_flatness', 'double_bond_flatness', 'internal_energy'],
    'Intermolecular Validity': ['minimum_distance_to_protein', 'minimum_distance_to_organic_cofactors', 
                              'minimum_distance_to_inorganic_cofactors', 'volume_overlap_with_protein',
                              'volume_overlap_with_organic_cofactors', 'volume_overlap_with_inorganic_cofactors']
}

# Calculate overall pass rate for each method
overall_pass_rates = pass_rates_df.mean()

# Create a bar chart for overall pass rates with better formatting
plt.figure(figsize=(12, 7))
# Sort methods by overall performance
sorted_methods = overall_pass_rates.sort_values(ascending=False).index
ax = sns.barplot(x=sorted_methods, y=overall_pass_rates[sorted_methods], palette='viridis')
plt.title('Overall PoseBusters Pass Rate (%) for Best Poses', fontsize=16)
plt.xlabel('Method', fontsize=14)
plt.ylabel('Average Pass Rate (%)', fontsize=14)
plt.xticks(rotation=45)
plt.ylim(0, 100)

# Add value labels on top of bars
for i, v in enumerate(overall_pass_rates[sorted_methods]):
    ax.text(i, v + 1, f'{v:.1f}%', ha='center', fontsize=11)

plt.grid(axis='y', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

# Plot category-specific pass rates
for category, tests in categories.items():
    category_pass_rates = pass_rates_df.loc[tests].mean()
    plt.figure(figsize=(12, 7))
    sorted_methods = category_pass_rates.sort_values(ascending=False).index
    ax = sns.barplot(x=sorted_methods, y=category_pass_rates[sorted_methods], palette='viridis')
    plt.title(f'Average PoseBusters Pass Rate (%) - {category}', fontsize=16)
    plt.xlabel('Method', fontsize=14)
    plt.ylabel('Average Pass Rate (%)', fontsize=14)
    plt.xticks(rotation=45)
    plt.ylim(0, 100)
    
    # Add value labels
    for i, v in enumerate(category_pass_rates[sorted_methods]):
        ax.text(i, v + 1, f'{v:.1f}%', ha='center', fontsize=11)
        
    plt.grid(axis='y', linestyle='--', alpha=0.3)
    plt.tight_layout()
    plt.show()

## Universal

In [ ]:
# List of methods to include in analysis
methods = ['diffdock_pocket_only', 'chai-1', 'icm', 'gnina', 'surfdock', 'vina']

# Run the universal analysis
universal_results = analysis.analyze_universal(
    methods=methods,
    rmsd_threshold=2.0,
    plot=True
)

# Access key results
success_proteins = universal_results['all_success_proteins']
failure_proteins = universal_results['all_failure_proteins']
significant_metrics = universal_results['significant_metrics']

### physics-based

In [ ]:
df_combined.shape, df_combined['protein'].nunique()

In [ ]:
# List of methods to include in analysis
methods = ['vina', 'icm']

# Run the universal analysis
universal_results = analysis.analyze_universal(
    methods=methods,
    rmsd_threshold=2.0,
    plot=True
)

# Access key results
success_proteins = universal_results['all_success_proteins']
failure_proteins = universal_results['all_failure_proteins']
significant_metrics = universal_results['significant_metrics']

#### ML-based

In [ ]:
# List of methods to include in analysis
methods = ['diffdock_pocket_only', 'chai-1', 'surfdock',]

# Run the universal analysis
universal_results = analysis.analyze_universal(
    methods=methods,
    rmsd_threshold=2.0,
    plot=True
)

# Access key results
success_proteins = universal_results['all_success_proteins']
failure_proteins = universal_results['all_failure_proteins']
significant_metrics = universal_results['significant_metrics']

## Individual

### ICM

In [ ]:
# Analyze a single method with default RMSD threshold of 2.0Å
single_method_results = analysis.analyze_single_method(
    method="icm",
    rmsd_threshold=2.0,
    imbalance_threshold=0.05,
    plot=True  # Generate visualization plots
)

# Access the results
success_rate = single_method_results['overall_success_rate']
significant_metrics = single_method_results['significant_metrics']

### gnina

In [ ]:
# Analyze a single method with default RMSD threshold of 2.0Å
single_method_results = analysis.analyze_single_method(
    method="gnina",
    rmsd_threshold=2.0,
    imbalance_threshold=0.05,
    plot=True  # Generate visualization plots
)

# Access the results
success_rate = single_method_results['overall_success_rate']
significant_metrics = single_method_results['significant_metrics']

### vina

In [ ]:
# Analyze a single method with default RMSD threshold of 2.0Å
single_method_results = analysis.analyze_single_method(
    method="vina",
    rmsd_threshold=2.0,
    imbalance_threshold=0.05,
    plot=True  # Generate visualization plots
)

# Access the results
success_rate = single_method_results['overall_success_rate']
significant_metrics = single_method_results['significant_metrics']

### surfdock

In [ ]:
# Analyze a single method with default RMSD threshold of 2.0Å
single_method_results = analysis.analyze_single_method(
    method="surfdock",
    rmsd_threshold=2.0,
    imbalance_threshold=0.05,
    plot=True  # Generate visualization plots
)

# Access the results
success_rate = single_method_results['overall_success_rate']
significant_metrics = single_method_results['significant_metrics']

### diffdock_pocket_only

In [ ]:
# Analyze a single method with default RMSD threshold of 2.0Å
single_method_results = analysis.analyze_single_method(
    method="diffdock_pocket_only",
    rmsd_threshold=2.0,
    imbalance_threshold=0.05,
    plot=True  # Generate visualization plots
)

# Access the results
success_rate = single_method_results['overall_success_rate']
significant_metrics = single_method_results['significant_metrics']

### chai-1

In [ ]:
# Analyze a single method with default RMSD threshold of 2.0Å
single_method_results = analysis.analyze_single_method(
    method="chai-1",
    rmsd_threshold=2.0,
    imbalance_threshold=0.05,
    plot=True  # Generate visualization plots
)

# Access the results
success_rate = single_method_results['overall_success_rate']
significant_metrics = single_method_results['significant_metrics']

## Comparative 

In [ ]:
comparative_results = analysis.analyze_comparative(
    method1="diffdock_pocket_only",
    method2="icm",
    rmsd_threshold=2.0,
    plot=True
)

# Access results
method1_only = comparative_results['method1_only_proteins']
method2_only = comparative_results['method2_only_proteins']
significant_metrics = comparative_results['significant_metrics']

In [ ]:
ml_methods = ['diffdock_pocket_only', 'surfdock', 'chai-1']
physics_methods = ['gnina', 'icm']

group_results = analysis.analyze_method_group_comparative(
    group1_methods=ml_methods,
    group2_methods=physics_methods,
    group1_label="ML-based",
    group2_label="Physics-based",
    rmsd_threshold=2.0,
    plot=True
)

### Comparative Mixed Effect

#### minimum_distance_to_protein

In [ ]:
analysis = PoseBustersAnalysis(df_combined)
result = analysis.mixed_effect_analysis(
    filter_name='minimum_distance_to_protein',
    method_groups={
        'ML-based': ['diffdock_pocket_only', 'chai-1', 'surfdock'], 
        'Physics-based': ['vina', 'gnina', 'icm']
    },
    rmsd_threshold=2.0,
    outcome_type='rmsd',
    plot=True
)
# Access model and results
# print("Model:", result['model'].summary())
# print("Interaction p-value:", result['p_interaction'])

In [ ]:
result = analysis.mixed_effect_analysis(
    filter_name="minimum_distance_to_protein",
    method_groups={
        'ML-based': ['diffdock_pocket_only', 'chai-1', 'surfdock'], 
        'Physics-based': ['vina', 'icm']
    },
    rmsd_threshold=2.0,
    outcome_type='rmsd',
    plot=True
)

#### all

In [ ]:
for prop in BUST_TEST_COLUMNS[1:]:
    print("Property:", prop)
    result = analysis.mixed_effect_analysis(
        filter_name=prop,
        method_groups={
            'ML-based': ['diffdock_pocket_only', 'chai-1', 'surfdock'], 
            'Physics-based': ['vina', 'icm']
        },
        rmsd_threshold=2.0,
        outcome_type='rmsd',
        plot=True
    )